In [0]:
pip install yfinance --quiet

In [0]:
import yfinance as yf
import pandas as pd

# Setar os parametros da consulta
ticker = 'BTC-USD'
periodo = '1y'
intervalo = '1d'

# Carregar os dados
dados_brutos = yf.download(ticker, period=periodo, interval=intervalo)

# Formato os dados
dados_brutos = dados_brutos.reset_index()
dados_brutos.columns = [coluna[0].lower() for coluna in dados_brutos.columns]

# Rename das colunas
dados_brutos = dados_brutos.rename(columns={
    'date': 'data',
    'open': 'abertura',
    'high': 'maxima',
    'low': 'minima',
    'close': 'fechamento',
    'volume': 'volume'
})

dados_brutos.head()

In [0]:
dados_brutos.shape

In [0]:
# tratamento para garantir formato dos dados na tabela
dados_brutos['data']       = pd.to_datetime(dados_brutos['data']).dt.date
dados_brutos['abertura']   = dados_brutos['abertura'].astype(float)
dados_brutos['maxima']     = dados_brutos['maxima'].astype(float)
dados_brutos['minima']     = dados_brutos['minima'].astype(float)
dados_brutos['fechamento'] = dados_brutos['fechamento'].astype(float)
dados_brutos['volume']     = dados_brutos['volume'].astype(float)

In [0]:
# Ingestão com o Spark
from pyspark.sql import SparkSession

# Cluster Spark
spark = SparkSession.builder.getOrCreate()

# converte o DF para Spark
df_spark = spark.createDataFrame(dados_brutos)

# insere na tabela já criada via SQL
(   
    df_spark
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('db_analytics.precos_brutos_bitcoin')
)

print(f'Total de registros inseridos: {df_spark.count()}')